### Normalization Formula

$$
x_{\text{normalized}} = \frac{x - \mu}{\sigma}
$$

Where:

- $x$ = pixel value after `ToTensor()` (between 0 and 1)
- $\mu$ = mean
- $\sigma$ = standard deviation

For our CIFAR-10 transformation:

$$
x_{\text{normalized}} = \frac{x - 0.5}{0.5}
$$

# 1. DATA LOADING & PREPROCESSING

In [1]:
import torch
import torchvision.transforms as transforms
import torchvision

batch = 64       # number of images processed before one weight update
learning_rate = 0.001 # controls how much the weights change in each update
epoch = 5             # number of times the entire training dataset is processed

CLASSES = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')  # 10 classes in CIFAR-10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # use GPU if CUDA is available, otherwise CPU

print(f"Using device: {device}")


transform = transforms.Compose([  # combines multiple transformations
    transforms.ToTensor(),        # converts image to tensor and pixel values from 0-255 to 0-1
    transforms.Normalize(
        (0.5, 0.5, 0.5),          # mean for R, G, B
        (0.5, 0.5, 0.5)           # standard deviation for R, G, B
    )                              # transforms values from 0-1 to roughly -1 to 1
])
# download the dataset
train_set=torchvision.datasets.CIFAR10(root='./data',train=
                                      True,download=True,transform=transform)# tranform tell what transofmration will be done,transformation is not done here
test_set=torchvision.datasets.CIFAR10(root='./data',train=
                                      False,download=True,transform=transform)
train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch,
    shuffle=True,
    num_workers=2
)  # loads images, applies the defined transformations, and sends them in batches to the neural network

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=batch,
    shuffle=False,
    num_workers=2
)  # shuffling is not required because the test data is only used for evaluation
print(f" Length of train set:{len(train_set)}\nLength of test set:{len(test_set)} ")

Using device: cuda


100%|████████████████████████████████████████████████████████████████████████████████| 170M/170M [26:13<00:00, 108kB/s]


Extracting ./data\cifar-10-python.tar.gz to ./data


C:\Users\lekal\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified


# 2. EXPLORATORY DATA ANALYSIS (EDA)
### Normalization and Undo Normalization

#### Normalization

The normalization formula is:

$$
x_{\text{normalized}} = \frac{x-\mu}{\sigma}
$$

For our case:

$$
\mu = 0.5,\qquad \sigma = 0.5
$$

Therefore:

$$
x_{\text{normalized}} = \frac{x-0.5}{0.5}
$$

---

### Undo Normalization

Start with:

$$
x_{\text{normalized}} = \frac{x-0.5}{0.5}
$$

Multiply both sides by $0.5$:

$$
0.5x_{\text{normalized}} = x-0.5
$$

Add $0.5$ to both sides:

$$
x = 0.5x_{\text{normalized}} + 0.5
$$

Since multiplying by $0.5$ is the same as dividing by $2$:

$$
x = \frac{x_{\text{normalized}}}{2} + 0.5
$$

Therefore, in Python:

    img = images[i] / 2 + 0.5

This converts the normalized pixel values from approximately $[-1,1]$ back to $[0,1]$ for displaying the image.

In [16]:
import numpy as np

def show_sample_images():
    images, label = next(iter(train_loader))  # gets one batch of images and labels

    fig, axes = plt.subplots(2, 5, figsize=(10, 4))

    for i, ax in enumerate(axes.flat):  # axes.flat lets us go through all 10 subplots one by one

        img = images[i] / 2 + 0.5  # converts normalized values (-1 to 1) back to scaled values (0 to 1)

        change_dimension = img.numpy().transpose(1, 2, 0)
        # converts Tensor → NumPy array
        # and changes (channel, height, width) → (height, width, channel)
        # because matplotlib expects this format

        ax.imshow(change_dimension)
        ax.set_title(CLASSES[label[i]])
        ax.axis('off')

    plt.tight_layout()  # adjusts spacing between subplots

    plt.savefig('sample_images.png')

    print("Saved sample_images.png - open it to see the sample training images")

    plt.close()  # closes the figure and frees its resources

show_sample_images()

Saved sample_images.png - open it to see the sample training images
